In [35]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re
from pathlib import Path

BASE_URL = "https://books.toscrape.com/"
NUM_PAGES = 5
GBP_TO_INR = 105.50

print("Setup completed successfully.")
print(f"GBP to INR conversion rate: {GBP_TO_INR}")
print(f"Pages to scrape: {NUM_PAGES}")

Setup completed successfully.
GBP to INR conversion rate: 105.5
Pages to scrape: 5


In [3]:
# Test connection to the data source

response = requests.get(BASE_URL, timeout=10)

print("Status code:", response.status_code)

if response.status_code == 200:
    print("Successfully connected to Books to Scrape.")
else:
    raise Exception(f"Failed to access website. Status code: {response.status_code}")

Status code: 200
Successfully connected to Books to Scrape.


In [4]:
# Scrape books from the first 5 catalogue pages

all_books = []

for page_num in range(1, NUM_PAGES + 1):

    if page_num == 1:
        url = BASE_URL
    else:
        url = f"{BASE_URL}catalogue/page-{page_num}.html"

    response = requests.get(url, timeout=10)

    if response.status_code != 200:
        print(f"Skipping page {page_num}: HTTP {response.status_code}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.select("article.product_pod")

    print(f"Page {page_num}: {len(books)} books found")

    for book in books:

        # Book title
        title_tag = book.select_one("h3 a")
        title = title_tag.get("title", "").strip() if title_tag else None

        # Price
        price_tag = book.select_one(".price_color")
        price = price_tag.get_text(strip=True) if price_tag else None

        # Star rating
        rating_tag = book.select_one(".star-rating")
        star_rating = (
            rating_tag.get("class")[-1]
            if rating_tag and len(rating_tag.get("class", [])) > 1
            else None
        )

        # Availability
        availability_tag = book.select_one(".availability")
        availability = (
            availability_tag.get_text(" ", strip=True)
            if availability_tag
            else None
        )

        # Category
        category = None

        if title_tag and title_tag.get("href"):
            book_url = requests.compat.urljoin(url, title_tag["href"])

            book_response = requests.get(book_url, timeout=10)

            if book_response.status_code == 200:
                book_soup = BeautifulSoup(book_response.text, "html.parser")

                breadcrumb = book_soup.select(
                    "ul.breadcrumb li a"
                )

                if len(breadcrumb) >= 3:
                    category = breadcrumb[-1].get_text(strip=True)

        all_books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

print("\nScraping completed.")
print("Total books scraped:", len(all_books))

Page 1: 20 books found
Page 2: 20 books found
Page 3: 20 books found
Page 4: 20 books found
Page 5: 20 books found

Scraping completed.
Total books scraped: 100


In [5]:
# Convert scraped data into a pandas DataFrame

books_df = pd.DataFrame(all_books)

print("Dataset shape:", books_df.shape)

display(books_df.head())

print("\nData types:")
print(books_df.dtypes)

print("\nMissing values:")
print(books_df.isnull().sum())

Dataset shape: (100, 5)


,title,price,star_rating,availability,category
0,A Light in the Attic,Â£51.77,Three,In stock,Poetry
1,Tipping the Velvet,Â£53.74,One,In stock,Historical Fiction
2,Soumission,Â£50.10,One,In stock,Fiction
3,Sharp Objects,Â£47.82,Four,In stock,Mystery
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,History



Data types:
title           object
price           object
star_rating     object
availability    object
category        object
dtype: object

Missing values:
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64


In [10]:
# Cleaning and transforming scraped data

# Making a copy so the original scraped DataFrame remains unchanged
cleaned_df = books_df.copy()


# 1. Clean price

def clean_price(value):
    try:
        text = str(value).strip()

        # Handle the encoding artifact "Â£"
        text = text.replace("Â£", "")

        # Also handle a normal pound symbol if it appears
        text = text.replace("£", "")

        return float(text)

    except (ValueError, TypeError):
        return None


cleaned_df["price_gbp"] = cleaned_df["price"].apply(clean_price)


# 2. Convert star rating text to integer

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

cleaned_df["rating"] = cleaned_df["star_rating"].map(rating_map)


# 3. Convert availability text to Boolean

def parse_availability(value):
    if pd.isna(value):
        return None

    text = str(value).strip().lower()

    if "in stock" in text:
        return True
    elif "out of stock" in text:
        return False
    else:
        return None


cleaned_df["in_stock"] = cleaned_df["availability"].apply(parse_availability)


# 4. Handle parsing failures

# Numeric price: median imputation
if cleaned_df["price_gbp"].isna().any():

    price_median = cleaned_df["price_gbp"].median()

    if pd.notna(price_median):
        cleaned_df["price_gbp"] = (
            cleaned_df["price_gbp"]
            .fillna(price_median)
        )

        print(
            f"Missing price values imputed using median: "
            f"{price_median:.2f}"
        )


# Numeric rating: median imputation
if cleaned_df["rating"].isna().any():

    rating_median = cleaned_df["rating"].median()

    if pd.notna(rating_median):
        cleaned_df["rating"] = (
            cleaned_df["rating"]
            .fillna(rating_median)
            .round()
            .astype(int)
        )

        print(
            f"Missing rating values imputed using median: "
            f"{rating_median}"
        )


# Availability: drop rows if it cannot be parsed
invalid_stock_rows = cleaned_df["in_stock"].isna().sum()

if invalid_stock_rows > 0:

    cleaned_df = cleaned_df.dropna(
        subset=["in_stock"]
    ).copy()

    print(
        f"Dropped {invalid_stock_rows} rows "
        f"with unparseable availability."
    )

cleaned_df["in_stock"] = cleaned_df["in_stock"].astype(bool)


# 5. GBP → INR converting with given rate

cleaned_df["price_inr"] = (
    cleaned_df["price_gbp"] * GBP_TO_INR
)


print("Superb! Cleaning and transformation completed.")

Superb! Cleaning and transformation completed.


In [17]:
# Final cleaned columns

final_df = cleaned_df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()

print("Final dataset shape:", final_df.shape)

display(final_df.head(10))

Final dataset shape: (100, 6)


,title,price_gbp,price_inr,rating,in_stock,category
0,A Light in the Attic,51.77,5461.735,3,True,Poetry
1,Tipping the Velvet,53.74,5669.570,1,True,Historical Fiction
2,Soumission,50.10,5285.550,1,True,Fiction
3,Sharp Objects,47.82,5045.010,4,True,Mystery
4,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,History
5,The Requiem Red,22.65,2389.575,1,True,Young Adult
6,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.370,4,True,Business
7,The Coming Woman: A Novel Based on the Life of...,17.93,1891.615,3,True,Default
8,The Boys in the Boat: Nine Americans and Their...,22.60,2384.300,4,True,Default
9,The Black Maria,52.15,5501.825,1,True,Poetry


In [19]:
final_df.isna().sum()

,0
title,0
price_gbp,0
price_inr,0
rating,0
in_stock,0
category,0


In [21]:
final_df.shape

(100, 6)

In [25]:
categories_df = (
    final_df[["category"]]
    .drop_duplicates()
    .sort_values("category")
    .reset_index(drop=True)
)

categories_df["category_id"] = range(
    1,
    len(categories_df) + 1
)

categories_df = categories_df[
    ["category_id", "category"]
].rename(
    columns={"category": "category_name"}
)

print("Number of categories:", len(categories_df))
display(categories_df)

Number of categories: 29


,category_id,category_name
0,1,Add a comment
1,2,Art
2,3,Business
3,4,Childrens
4,5,Contemporary
5,6,Default
6,7,Fantasy
7,8,Fiction
8,9,Food and Drink
9,10,Health


In [33]:
category_mapping = dict(
    zip(
        categories_df["category_name"],
        categories_df["category_id"]
    )
)

books_db_df = final_df.copy()

books_db_df["category_id"] = (
    books_db_df["category"]
    .map(category_mapping)
)

books_db_df.insert(
    0,
    "book_id",
    range(1, len(books_db_df) + 1)
)

print("Books table shape:", books_db_df.shape)
print("Missing category IDs:", books_db_df["category_id"].isna().sum())

display(books_db_df.head())

books_sql_df = books_db_df[
    [
        "book_id",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id"
    ]
].copy()

Books table shape: (100, 8)
Missing category IDs: 0


,book_id,title,price_gbp,price_inr,rating,in_stock,category,category_id
0,1,A Light in the Attic,51.77,5461.735,3,True,Poetry,19
1,2,Tipping the Velvet,53.74,5669.570,1,True,Historical Fiction,11
2,3,Soumission,50.10,5285.550,1,True,Fiction,8
3,4,Sharp Objects,47.82,5045.010,4,True,Mystery,15
4,5,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,History,12


In [34]:
print("Categories:")
print(categories_df.shape)
display(categories_df.head())

print("\nBooks:")
print(books_sql_df.shape)
display(books_sql_df.head())

print("\nMissing category IDs:")
print(books_sql_df["category_id"].isna().sum())

print("\nUnique category IDs in books:")
print(sorted(books_sql_df["category_id"].unique()))

Categories:
(29, 2)


,category_id,category_name
0,1,Add a comment
1,2,Art
2,3,Business
3,4,Childrens
4,5,Contemporary



Books:
(100, 7)


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,A Light in the Attic,51.77,5461.735,3,True,19
1,2,Tipping the Velvet,53.74,5669.570,1,True,11
2,3,Soumission,50.10,5285.550,1,True,8
3,4,Sharp Objects,47.82,5045.010,4,True,15
4,5,Sapiens: A Brief History of Humankind,54.23,5721.265,5,True,12



Missing category IDs:
0

Unique category IDs in books:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29)]


In [37]:
# Create SQLite database

import sqlite3
from pathlib import Path


db_path = Path("zepto_books.db")
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys = ON")

print(f"SQLite database created at: {db_path.resolve()}")

SQLite database created at: /content/zepto_books.db


In [38]:
# Create normalized categories and books tables

create_tables_sql = """
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
);

CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL,
    in_stock INTEGER NOT NULL,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);
"""

conn.executescript(create_tables_sql)
conn.commit()

print("Tables created successfully.")

Tables created successfully.


In [39]:
# Verifying database tables

tables_df = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

display(tables_df)

,name
0,books
1,categories


In [40]:
# Insert categories into the categories table

categories_to_sql = categories_df[
    [
        "category_id",
        "category_name"
    ]
].copy()

categories_to_sql.to_sql(
    "categories",
    conn,
    if_exists="append",
    index=False
)

print("Categories inserted:", len(categories_to_sql))

Categories inserted: 29


In [41]:
# Insert books into the books table

books_sql_df.to_sql(
    "books",
    conn,
    if_exists="append",
    index=False
)

print("Books inserted:", len(books_sql_df))

Books inserted: 100


In [42]:
# Verify row counts in both tables

categories_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM categories;",
    conn
)

books_count = pd.read_sql(
    "SELECT COUNT(*) AS count FROM books;",
    conn
)

print("Categories in database:")
display(categories_count)

print("Books in database:")
display(books_count)

Categories in database:


,count
0,29


Books in database:


,count
0,100


In [43]:
# Verify the foreign-key relationship

relationship_check = pd.read_sql(
    """
    SELECT
        b.book_id,
        b.title,
        b.category_id,
        c.category_name
    FROM books b
    JOIN categories c
        ON b.category_id = c.category_id
    LIMIT 10;
    """,
    conn
)

display(relationship_check)

,book_id,title,category_id,category_name
0,1,A Light in the Attic,19,Poetry
1,2,Tipping the Velvet,11,Historical Fiction
2,3,Soumission,8,Fiction
3,4,Sharp Objects,15,Mystery
4,5,Sapiens: A Brief History of Humankind,12,History
5,6,The Requiem Red,29,Young Adult
6,7,The Dirty Little Secrets of Getting Your Dream...,3,Business
7,8,The Coming Woman: A Novel Based on the Life of...,6,Default
8,9,The Boys in the Boat: Nine Americans and Their...,6,Default
9,10,The Black Maria,19,Poetry


In [44]:
# Check the foreign-key definition

foreign_keys = pd.read_sql(
    "PRAGMA foreign_key_list(books);",
    conn
)

display(foreign_keys)

,id,seq,table,from,to,on_update,on_delete,match
0,0,0,categories,category_id,category_id,NO ACTION,NO ACTION,NONE


In [51]:
# SQL queries

queries = {

    "query_1_select_where": """
        SELECT
            title,
            price_gbp,
            rating,
            in_stock
        FROM books
        WHERE rating >= 4;
    """,

    "query_2_order_by": """
        SELECT
            title,
            price_gbp
        FROM books
        ORDER BY price_gbp DESC;
    """,

    "query_3_limit": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10;
    """,

    "query_4_distinct": """
        SELECT DISTINCT
            rating
        FROM books
        ORDER BY rating;
    """,

    "query_5_between": """
        SELECT
            title,
            price_gbp,
            rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp;
    """,

    "query_6_join": """
        SELECT
            b.title,
            c.category_name,
            b.rating,
            b.price_gbp,
            b.in_stock
        FROM books AS b
        JOIN categories AS c
            ON b.category_id = c.category_id
        ORDER BY b.rating DESC, b.price_gbp DESC
        LIMIT 10;
    """
}

print("Total SQL queries:", len(queries))

Total SQL queries: 6


In [52]:
# Executing all required SQL queries and displaying their outputs

query_results = {}

for query_name, query in queries.items():

    print("=" * 80)
    print(query_name.upper())
    print("=" * 80)

    print("SQL:")
    print(query.strip())

    result = pd.read_sql(query, conn)

    query_results[query_name] = result

    print("\nOUTPUT:")
    display(result)

    print()

QUERY_1_SELECT_WHERE
SQL:
SELECT
            title,
            price_gbp,
            rating,
            in_stock
        FROM books
        WHERE rating >= 4;

OUTPUT:


,title,price_gbp,rating,in_stock
0,Sharp Objects,47.82,4,1
1,Sapiens: A Brief History of Humankind,54.23,5,1
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4,1
3,The Boys in the Boat: Nine Americans and Their...,22.60,4,1
4,Shakespeare's Sonnets,20.66,4,1
5,Set Me Free,17.46,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5,1
7,Rip it Up and Start Again,35.02,5,1
8,Chase Me (Paris Nights #2),25.27,5,1
9,Black Dust,34.53,5,1



QUERY_2_ORDER_BY
SQL:
SELECT
            title,
            price_gbp
        FROM books
        ORDER BY price_gbp DESC;

OUTPUT:


,title,price_gbp
0,The Death of Humanity: and the Case for Life,58.11
1,Slow States of Collapse: Poems,57.31
2,Our Band Could Be Your Life: Scenes from the A...,57.25
3,The Past Never Ends,56.50
4,The Pioneer Woman Cooks: Dinnertime: Comfort C...,56.41
...,...,...
95,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99
96,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61
97,Princess Between Worlds (Wide-Awake Princess #5),13.34
98,In Her Wake,12.84



QUERY_3_LIMIT
SQL:
SELECT
            title,
            price_gbp,
            rating
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10;

OUTPUT:


...

,rating
0,1
1,2
2,3
3,4
4,5



QUERY_5_BETWEEN
SQL:
SELECT
            title,
            price_gbp,
            rating
        FROM books
        WHERE price_gbp BETWEEN 20 AND 40
        ORDER BY price_gbp;

OUTPUT:


,title,price_gbp,rating
0,The Inefficiency Assassin: Time Management Tac...,20.59,5
1,Shakespeare's Sonnets,20.66,4
2,In the Country We Love: My Family Divided,22.00,4
3,America's Cradle of Quarterbacks: Western Penn...,22.50,3
4,The Boys in the Boat: Nine Americans and Their...,22.60,4
5,The Requiem Red,22.65,1
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5
7,The Elephant Tree,23.82,5
8,Olio,23.88,1
9,The Mindfulness and Acceptance Workbook for An...,23.89,4



QUERY_6_JOIN
SQL:
SELECT
            b.title,
            c.category_name,
            b.rating,
            b.price_gbp,
            b.in_stock
        FROM books AS b
        JOIN categories AS c
            ON b.category_id = c.category_id
        ORDER BY b.rating DESC, b.price_gbp DESC
        LIMIT 10;

OUTPUT:


,title,category_name,rating,price_gbp,in_stock
0,Sapiens: A Brief History of Humankind,History,5,54.23,1
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,1
2,"We Love You, Charlie Freeman",Fiction,5,50.27,1
3,Private Paris (Private #10),Fiction,5,47.61,1
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,5,40.30,1
5,Join,Science Fiction,5,35.67,1
6,Rip it Up and Start Again,Music,5,35.02,1
7,Black Dust,Romance,5,34.53,1
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,5,32.24,1
9,Chase Me (Paris Nights #2),Romance,5,25.27,1


In [64]:
# Read SQL query results into pandas DataFrames

sql_result_where = pd.read_sql(
    queries["query_1_select_where"],
    conn
)

sql_result_join = pd.read_sql(
    queries["query_6_join"],
    conn
)

print("Result from SELECT + WHERE query:")
display(sql_result_where.head(10))

print("\nResult from JOIN query:")
display(sql_result_join)

Result from SELECT + WHERE query:


,title,price_gbp,rating,in_stock
0,Sharp Objects,47.82,4,1
1,Sapiens: A Brief History of Humankind,54.23,5,1
2,The Dirty Little Secrets of Getting Your Dream...,33.34,4,1
3,The Boys in the Boat: Nine Americans and Their...,22.60,4,1
4,Shakespeare's Sonnets,20.66,4,1
5,Set Me Free,17.46,5,1
6,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5,1
7,Rip it Up and Start Again,35.02,5,1
8,Chase Me (Paris Nights #2),25.27,5,1
9,Black Dust,34.53,5,1



Result from JOIN query:


,title,category_name,rating,price_gbp,in_stock
0,Sapiens: A Brief History of Humankind,History,5,54.23,1
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,1
2,"We Love You, Charlie Freeman",Fiction,5,50.27,1
3,Private Paris (Private #10),Fiction,5,47.61,1
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,5,40.30,1
5,Join,Science Fiction,5,35.67,1
6,Rip it Up and Start Again,Music,5,35.02,1
7,Black Dust,Romance,5,34.53,1
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,5,32.24,1
9,Chase Me (Paris Nights #2),Romance,5,25.27,1


In [59]:
# SQL JOIN
sql_join_comparison = pd.read_sql(
    queries["query_6_join"],
    conn
).reset_index(drop=True)

print("SQL JOIN result:")
display(sql_join_comparison)

SQL JOIN result:


,title,category_name,rating,price_gbp,in_stock
0,Sapiens: A Brief History of Humankind,History,5,54.23,1
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,1
2,"We Love You, Charlie Freeman",Fiction,5,50.27,1
3,Private Paris (Private #10),Fiction,5,47.61,1
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,5,40.30,1
5,Join,Science Fiction,5,35.67,1
6,Rip it Up and Start Again,Music,5,35.02,1
7,Black Dust,Romance,5,34.53,1
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,5,32.24,1
9,Chase Me (Paris Nights #2),Romance,5,25.27,1


In [66]:
# Now with using pandas merge

pandas_join_result = pd.merge(
    books_sql_df,
    categories_df,
    on="category_id",
    how="inner"
)

# Select the same columns and apply the same ordering
pandas_join_result = pandas_join_result[
    [
        "title",
        "category_name",
        "rating",
        "price_gbp",
        "in_stock"
    ]
].sort_values(
    by=["rating", "price_gbp"],
    ascending=[False, False]
).head(10).reset_index(drop=True)

print("Pandas merge result:")
display(pandas_join_result)

Pandas merge result:


,title,category_name,rating,price_gbp,in_stock
0,Sapiens: A Brief History of Humankind,History,5,54.23,True
1,Scott Pilgrim's Precious Little Life (Scott Pi...,Sequential Art,5,52.29,True
2,"We Love You, Charlie Freeman",Fiction,5,50.27,True
3,Private Paris (Private #10),Fiction,5,47.61,True
4,Worlds Elsewhere: Journeys Around Shakespeareâ...,Nonfiction,5,40.30,True
5,Join,Science Fiction,5,35.67,True
6,Rip it Up and Start Again,Music,5,35.02,True
7,Black Dust,Romance,5,34.53,True
8,The Activist's Tao Te Ching: Ancient Advice fo...,Spirituality,5,32.24,True
9,Chase Me (Paris Nights #2),Romance,5,25.27,True


In [65]:
# Normalize SQLite's integer representation of Boolean
sql_join_comparison["in_stock"] = (
    sql_join_comparison["in_stock"].astype(bool)
)

# Make sure both DataFrames have the same column order
pandas_join_comparison = pandas_join_result[
    [
        "title",
        "category_name",
        "rating",
        "price_gbp",
        "in_stock"
    ]
].reset_index(drop=True)

sql_join_comparison = sql_join_comparison[
    [
        "title",
        "category_name",
        "rating",
        "price_gbp",
        "in_stock"
    ]
].reset_index(drop=True)

### SQL JOIN vs Pandas Merge

The JOIN query is first executed against the SQLite database using `pd.read_sql()`.

The same relational operation is then reproduced in pandas using `pd.merge()` on `category_id`. The pandas DataFrame is built from the same normalized columns that were loaded into the SQLite `books` table.

SQLite represents Boolean values as integers, so the SQL result is converted back to Boolean before comparison.

Both outputs are aligned by column order and sorted identically. The equality check confirms that both approaches produce equivalent results.

In [67]:
joins_match = sql_join_comparison.equals(
    pandas_join_comparison
)

print("Do SQL JOIN and pandas merge produce equivalent results?")
print(joins_match)

Do SQL JOIN and pandas merge produce equivalent results?
True


In [68]:
# Final database verification

print("Database path:")
print(db_path.resolve())

print("\nDatabase tables:")
tables = pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)
display(tables)

print("\nCategories count:")
display(
    pd.read_sql(
        "SELECT COUNT(*) AS category_count FROM categories;",
        conn
    )
)

print("\nBooks count:")
display(
    pd.read_sql(
        "SELECT COUNT(*) AS book_count FROM books;",
        conn
    )
)

Database path:
/content/zepto_books.db

Database tables:


,name
0,books
1,categories



Categories count:


,category_count
0,29



Books count:


,book_count
0,100


In [69]:
# Verifying data stored in the books table

database_sample = pd.read_sql(
    """
    SELECT
        b.book_id,
        b.title,
        b.price_gbp,
        b.price_inr,
        b.rating,
        b.in_stock,
        c.category_name
    FROM books AS b
    JOIN categories AS c
        ON b.category_id = c.category_id
    ORDER BY b.book_id
    LIMIT 10;
    """,
    conn
)

display(database_sample)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_name
0,1,A Light in the Attic,51.77,5461.735,3,1,Poetry
1,2,Tipping the Velvet,53.74,5669.570,1,1,Historical Fiction
2,3,Soumission,50.10,5285.550,1,1,Fiction
3,4,Sharp Objects,47.82,5045.010,4,1,Mystery
4,5,Sapiens: A Brief History of Humankind,54.23,5721.265,5,1,History
5,6,The Requiem Red,22.65,2389.575,1,1,Young Adult
6,7,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.370,4,1,Business
7,8,The Coming Woman: A Novel Based on the Life of...,17.93,1891.615,3,1,Default
8,9,The Boys in the Boat: Nine Americans and Their...,22.60,2384.300,4,1,Default
9,10,The Black Maria,52.15,5501.825,1,1,Poetry


In [75]:
# Verifying the required fixed GBP → INR conversion

conversion_check = pd.read_sql(
    """
    SELECT
        title,
        price_gbp,
        price_inr,
        ROUND(price_gbp * 105.50, 2) AS expected_price_inr
    FROM books
    LIMIT 10;
    """,
    conn
)

conversion_check["matches_required_rate"] = (
    conversion_check["price_inr"].round(2)
    == conversion_check["expected_price_inr"].round(2)
)

display(conversion_check)

,title,price_gbp,price_inr,expected_price_inr,matches_required_rate
0,A Light in the Attic,51.77,5461.735,5461.74,True
1,Tipping the Velvet,53.74,5669.570,5669.57,True
2,Soumission,50.10,5285.550,5285.55,True
3,Sharp Objects,47.82,5045.010,5045.01,True
4,Sapiens: A Brief History of Humankind,54.23,5721.265,5721.27,False
5,The Requiem Red,22.65,2389.575,2389.58,False
6,The Dirty Little Secrets of Getting Your Dream...,33.34,3517.370,3517.37,True
7,The Coming Woman: A Novel Based on the Life of...,17.93,1891.615,1891.62,True
8,The Boys in the Boat: Nine Americans and Their...,22.60,2384.300,2384.30,True
9,The Black Maria,52.15,5501.825,5501.83,False


In [76]:
conn.close()

print("SQLite connection closed.")

SQLite connection closed.


In [77]:
# Verifying the SQLite database file exists

print("Database exists:", db_path.exists())
print("Database size:", db_path.stat().st_size, "bytes")

Database exists: True
Database size: 24576 bytes


# Module 1 Summary

This module implements an end-to-end data engineering pipeline for catalogue data.

The pipeline:

1. Scrapes 100 books from the first five pages of Books to Scrape.
2. Extracts title, price, star rating, availability, and category.
3. Cleans and converts the raw fields into appropriate data types.
4. Converts GBP prices to INR using the fixed project rate of 1 GBP = 105.50 INR.
5. Stores the cleaned data in a normalized SQLite database containing `categories` and `books` tables.
6. Executes six SQL queries demonstrating filtering, ordering, limiting, distinct values, range filtering, and table joins.
7. Reads SQL results into pandas using `pd.read_sql()`.
8. Reproduces the SQL JOIN using `pd.merge()`.
9. Confirms that the SQL JOIN and pandas merge produce equivalent results.

Final dataset size: **100 books across 29 categories**.